# Apply on ABDDs demonstration 4


In [ ]:
import itertools
from typing import Optional

from apply.abdd_apply_main import abdd_apply
from apply.abdd import convert_ta_to_abdd
from apply.box_algebra.apply_tables import BooleanOperation
from apply.abdd_node_cache import ABDDNodeCacheClass
from apply.abdd import ABDD, construct_node
from apply.abdd_node import ABDDNode
from apply.evaluation import compare_op_abdd, compare_abdds_tas

from formats.format_vtf import import_treeaut_from_vtf
from formats.render_dot import convert_to_dot
from helpers.utils import box_orders
from helpers.string_manipulation import create_var_order_list

from canonization.unfolding import ubda_unfolding
from canonization.normalization import ubda_normalize
from canonization.folding import ubda_folding
from tree_automata.functions.trimming import remove_useless_states

In [ ]:
def canonize(abdd: ABDD) -> ABDD:
    varcount = abdd.variable_count
    ta = abdd.convert_to_treeaut_obj()
    ta.reformat_states()
    unf = ubda_unfolding(ta, varcount+1)
    norm = ubda_normalize(unf, create_var_order_list("", varcount+1), fix=True)
    norm = remove_useless_states(norm)
    norm.reformat_states()
    fold = ubda_folding(norm, box_orders['full'], varcount+1)
    fold.remove_self_loops()
    fold = remove_useless_states(fold)
    canonabdd = convert_ta_to_abdd(fold, ncache=ABDDNodeCacheClass())
    return canonabdd

In [ ]:
op = BooleanOperation.AND

ncache = ABDDNodeCacheClass()
zero = ncache.terminal_0
one = ncache.terminal_1
varcount = 10

d1_8 = construct_node(8, "X", [zero], "X", [one], ncache)
d1_5 = construct_node(5, "X", [zero], "X", [d1_8], ncache)
d1_2 = construct_node(2, "X", [d1_5], "X", [one], ncache)
ncache.refresh_nodes()
abdd1 = ABDD('test1', varcount, [d1_2], rootrule="X")
convert_to_dot(abdd1)


In [ ]:
d2_10 = construct_node(10, None, [one], None, [zero], ncache)
d2_6 = construct_node(6, "X", [d2_10], "X", [one], ncache)
d2_3 = construct_node(3, "X", [zero], "X", [d2_6], ncache)
ncache.refresh_nodes()
abdd2 = ABDD('test2', varcount, [d2_3], rootrule="X")
convert_to_dot(abdd2)

In [ ]:
abdd1_c = canonize(abdd1)
convert_to_dot(abdd1_c)

In [ ]:
abdd2_c = canonize(abdd2)
convert_to_dot(abdd2_c)

In [ ]:
bdd_test = abdd_apply(op, abdd1, abdd2, cache=ABDDNodeCacheClass(), maxvar=varcount)
convert_to_dot(bdd_test)

In [ ]:
ncache = ABDDNodeCacheClass()
abdd_test = abdd_apply(op, abdd1_c, abdd2_c, ncache, maxvar=varcount)
ncache.refresh_nodes()
abdd_test.reformat_node_names()
abdd_test_c = canonize(abdd_test)
print('initial inputs match result:', compare_op_abdd(abdd1, abdd2, op, abdd_test_c))
print('canonized inputs match result:', compare_op_abdd(abdd1_c, abdd2_c, op, abdd_test_c))
print('result after canonization is consistent:', compare_abdds_tas(abdd_test_c, abdd_test))
convert_to_dot(abdd_test_c)

In [ ]:
ncache = ABDDNodeCacheClass()
abdd_test2 = abdd_apply(op, abdd1, abdd2, ncache, maxvar=varcount)
ncache.refresh_nodes()
abdd_test2.reformat_node_names()
abdd_test2_c = canonize(abdd_test2)
print('outputs have the same semantics:', compare_abdds_tas(abdd_test_c, abdd_test2_c))
print('outputs are isomorphic:', abdd_test_c == abdd_test2_c)
convert_to_dot(abdd_test2_c)